In [1]:
import ollama

In [2]:
def get_completion_from_messages(messages, model="llama3.2", temperature=0, max_tokens=500):
    response = ollama.chat(model=model, messages=messages, options={"temperature": temperature, "num_predict": max_tokens})
    return response["message"]["content"]

In [3]:
final_response_to_customer = """
Aashirvaad Atta is a high-quality whole wheat flour with a 5kg pack,
high fiber, and a rating of 4.5. It is priced at ₹250.

Aavin Milk is fresh milk available in a 500ml pack and is priced at ₹50.

For grocery products, we also have Tata Salt. Tata Salt is iodized salt
available in a 1kg pack, with a rating of 4.6 and a price of ₹30.
"""

In [4]:
moderation_prompt = f"""
You are a supermarket customer service assistant. Analyze the following customer message and determine whether 
it contains harmful, offensive, abusive, or unsafe content.
Respond with a single letter:
Y - if it contains harmful content
N - if it is safe.

Text: '''{final_response_to_customer}'''
"""

moderation_response = get_completion_from_messages([{"role": "user", "content": moderation_prompt}])
print("Moderation Decision:", moderation_response)

Moderation Decision: N


In [5]:
system_message = """
You are an evaluator for a supermarket customer service assistant.

Your task is to evaluate the Agent response using ONLY the
Customer message and Product information provided.

Return exactly one character: Y or N.

Return Y ONLY when BOTH conditions are true:

1. The Agent response answers all the main requests in the Customer message.
2. Every product fact stated in the Agent response is supported by
   the provided Product information.

Return N if either condition is false.

Do not use outside knowledge.

For example:
- If the Agent mentions a product that is NOT present in the
  Product information, return N.
- If the Agent gives a price that does not match the Product information,
  return N.
- If the Agent correctly discusses all requested products and the
  information matches, return Y.

Output only Y or N.
"""

customer_message = """
Tell me about Aashirvaad Atta and Aavin Milk.
Also tell me about your grocery products.
"""

product_information = """{
"name": "Aashirvaad Atta",
"category": "Grocery",
"brand": "Aashirvaad",
"rating": 4.5,
"features": ["5kg pack", "Whole wheat", "High fiber"],
"description": "High-quality whole wheat flour for everyday cooking.",
"price": 250
}
{
"name": "Aavin Milk",
"category": "Dairy and Beverages",
"brand": "Aavin",
"rating": 4.5,
"features": ["500ml pack", "Fresh milk"],
"description": "Fresh milk suitable for daily consumption.",
"price": 50
}
{
"name": "Tata Salt",
"category": "Grocery",
"brand": "Tata",
"rating": 4.6,
"features": ["1kg pack", "Iodized salt"],
"description": "Iodized salt for everyday cooking.",
"price": 30
}"""

q_a_pair = f"""
Customer message: ```{customer_message}```
Product information: ```{product_information}```
Agent response: ```{final_response_to_customer}```

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": q_a_pair}
]

response = get_completion_from_messages(messages, max_tokens=20)
print(response)

Y


In [6]:
another_response = "life is like a box of chocolates"
q_a_pair = f"""
Customer message: ```{customer_message}```
Product information: ```{product_information}```
Agent response: ```{another_response}```

Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": q_a_pair}
]

response = get_completion_from_messages(messages)
print(response)


N


In [ ]:
#Ollama API does not have a built-in moderation endpoint like OpenAI's Moderation API.